# Society MAS for a Toy Travel Agency

This notebook shows a **society-style multi-agent architecture**.
The agents do not follow a manager. Instead, they follow simple social rules and converge through voting.

## What Makes This a Society?

We still solve the same travel task, but the orchestration is different.
Instead of building the itinerary step by step, the society evaluates a few travel packages and chooses one through a tiny public voting process.

This lets us highlight:

- equal social status,
- public visibility of decisions,
- consensus or majority-based convergence.

In [1]:
from typing import Literal

from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

# One small model keeps the four tutorials easy to compare.
# The notebooks assume OPENAI_API_KEY is already set in your environment.
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

USER_REQUEST = """
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan
""".strip()

DESTINATIONS = {
    "Lisbon": {
        "best_period": "April to June",
        "style": "sunny, walkable, relaxed",
        "notes": "great for food, viewpoints, and compact neighborhoods",
    },
    "Barcelona": {
        "best_period": "April to June",
        "style": "lively, artistic, seaside",
        "notes": "strong mix of architecture, beach walks, and tapas",
    },
    "Prague": {
        "best_period": "April to June",
        "style": "historic, compact, lower-cost",
        "notes": "easy sightseeing with a classic old-town atmosphere",
    },
}

FLIGHTS = [
    {"destination": "Lisbon", "code": "TP-833", "price": 180, "type": "direct", "duration": "3h 05m"},
    {"destination": "Lisbon", "code": "IB-310", "price": 150, "type": "1 stop", "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct", "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop", "duration": "4h 00m"},
    {"destination": "Prague", "code": "FR-721", "price": 110, "type": "direct", "duration": "1h 55m"},
    {"destination": "Prague", "code": "OS-410", "price": 145, "type": "1 stop", "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon", "name": "Baixa Stay", "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon", "name": "River Rooms", "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel", "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn", "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague", "name": "Old Town House", "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague", "name": "City Garden Hotel", "price_per_night": 95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon", "name": "Alfama food walk", "tag": "food", "price": 35},
    {"destination": "Lisbon", "name": "Belem and river tram day", "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening", "tag": "food", "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Familia and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague", "name": "Old Town walking tour", "tag": "culture", "price": 18},
    {"destination": "Prague", "name": "Czech dinner and jazz night", "tag": "food", "price": 30},
]


def flights_for(destination: str) -> list[dict]:
    return [item for item in FLIGHTS if item["destination"] == destination]



def hotels_for(destination: str) -> list[dict]:
    return [item for item in HOTELS if item["destination"] == destination]



def activities_for(destination: str) -> list[dict]:
    return [item for item in ACTIVITIES if item["destination"] == destination]



def catalog_text() -> str:
    lines = []
    for destination, info in DESTINATIONS.items():
        lines.append(f"Destination: {destination}")
        lines.append(f"  Best period: {info['best_period']}")
        lines.append(f"  Style: {info['style']}")
        lines.append(f"  Notes: {info['notes']}")
        lines.append("  Flights:")
        for flight in flights_for(destination):
            lines.append(
                f"    - {flight['code']} | {flight['type']} | EUR {flight['price']} | {flight['duration']}"
            )
        lines.append("  Hotels:")
        for hotel in hotels_for(destination):
            lines.append(
                f"    - {hotel['name']} | EUR {hotel['price_per_night']} per night | {hotel['style']}"
            )
        lines.append("  Activities:")
        for activity in activities_for(destination):
            lines.append(
                f"    - {activity['name']} | {activity['tag']} | EUR {activity['price']}"
            )
        lines.append("")
    return "\n".join(lines).strip()


CATALOG_TEXT = catalog_text()
print(USER_REQUEST)

Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan


## Define the Citizens and the Voting Rule

First we convert the catalog into a few candidate packages.
Then each citizen votes according to a different social preference, such as budget, comfort, or culture.

The important idea is that **coordination happens through the public board**, not through a manager.

In [2]:
from collections import Counter


def build_package(destination: str) -> dict:
    # We build one simple candidate package per destination.
    cheapest_flight = sorted(flights_for(destination), key=lambda item: item["price"])[0]
    central_hotel = sorted(hotels_for(destination), key=lambda item: item["price_per_night"])[0]
    chosen_activities = activities_for(destination)[:2]

    return {
        "destination": destination,
        "period": DESTINATIONS[destination]["best_period"],
        "flight": cheapest_flight,
        "hotel": central_hotel,
        "activities": chosen_activities,
    }


PACKAGES = {
    "package_1": build_package("Lisbon"),
    "package_2": build_package("Barcelona"),
    "package_3": build_package("Prague"),
}


def packages_text() -> str:
    lines = []
    for package_id, package in PACKAGES.items():
        lines.append(f"{package_id}: {package['destination']}")
        lines.append(f"  Period: {package['period']}")
        lines.append(
            f"  Flight: {package['flight']['code']} | {package['flight']['type']} | EUR {package['flight']['price']}"
        )
        lines.append(
            f"  Hotel: {package['hotel']['name']} | EUR {package['hotel']['price_per_night']} per night"
        )
        lines.append(
            "  Activities: " + ", ".join(item["name"] for item in package["activities"])
        )
        lines.append("")
    return "\n".join(lines).strip()


PACKAGES_TEXT = packages_text()


class Vote(BaseModel):
    package_id: Literal["package_1", "package_2", "package_3"]
    reason: str = Field(description="Why this citizen picked the package.")
    confidence: int = Field(description="Confidence from 1 to 5.", ge=1, le=5)


voter = model.with_structured_output(Vote)

CITIZENS = {
    "Budget citizen": "Prefer the lowest reasonable total cost.",
    "Comfort citizen": "Prefer direct flights and the easiest overall trip.",
    "Culture citizen": "Prefer the strongest mix of food and cultural activities.",
    "Simplicity citizen": "Prefer the most coherent 4-day plan with minimal friction.",
}


def cast_vote(citizen_name: str, rule: str, public_board: str) -> Vote:
    # Each citizen sees the same public board and follows one social rule.
    # No manager tells them what to do.
    return voter.invoke(
        [
            SystemMessage(
                content=(
                    "You are a citizen in a society-style travel MAS. "
                    "Nobody manages the group. Follow your social rule, inspect the public board, and cast one vote.\n\n"
                    f"Your social rule: {rule}"
                )
            ),
            HumanMessage(
                content=(
                    f"Traveler request:\n{USER_REQUEST}\n\n"
                    f"Candidate packages:\n{PACKAGES_TEXT}\n\n"
                    f"Public board:\n{public_board or 'No votes yet.'}\n\n"
                    "Vote for exactly one package."
                )
            ),
        ]
    )

## Run the Society

We use a tiny two-round voting loop.
After each round, the public board becomes part of the context for the next round.
If one package gets a majority, the society stops early.

In [3]:
public_board = []
latest_count = Counter()
winning_package = None

# The society uses a tiny consensus loop.
# After each round, everyone can see the public voting history.
for round_number in range(2):
    latest_count = Counter()

    for citizen_name, rule in CITIZENS.items():
        ballot = cast_vote(citizen_name, rule, "\n".join(public_board))
        public_board.append(
            f"Round {round_number + 1} | {citizen_name} voted for {ballot.package_id} "
            f"(confidence {ballot.confidence}): {ballot.reason}"
        )
        latest_count[ballot.package_id] += 1

    # Majority of 3 out of 4 is enough to stop.
    best_package, votes = latest_count.most_common(1)[0]
    if votes >= 3:
        winning_package = best_package
        break

if winning_package is None:
    winning_package = latest_count.most_common(1)[0][0]

final_summary = model.invoke(
    [
        SystemMessage(
            content=(
                "Explain the final package chosen by the society. "
                "Mention the consensus logic in plain language."
            )
        ),
        HumanMessage(
            content=(
                f"Traveler request:\n{USER_REQUEST}\n\n"
                f"Public board:\n{'\n'.join(public_board)}\n\n"
                f"Winning package:\n{PACKAGES[winning_package]}"
            )
        ),
    ]
).content

print("=== Public board ===")
for item in public_board:
    print("-", item)

print("\n=== Winning package ===")
print(PACKAGES[winning_package])

print("\n=== Final summary ===")
print(final_summary)

=== Public board ===
- Round 1 | Budget citizen voted for package_3 (confidence 5): Package 3 offers the lowest total cost with direct flights and a reasonably priced hotel, fitting the mid-range budget. It also includes a good mix of food and culture with simple daily plans, meeting all requirements effectively.
- Round 1 | Comfort citizen voted for package_3 (confidence 5): Package 3 has a direct flight, which aligns with my preference for easy flights. The hotel is centrally located and affordable, and the activities offer a good mix of food and culture with a simple plan. Overall, it is the easiest and most straightforward trip.
- Round 1 | Culture citizen voted for package_3 (confidence 5): Package 3 offers a direct flight, which is easier and more convenient. The hotel is centrally located and affordable, and the activities provide a strong mix of cultural experiences and food, matching the preference for a strong combination of food and culture with a simple daily plan.
- Round 

## Why This Fits the Society Architecture

- Agents follow simple social rules.
- The state is public and collective.
- The final answer emerges from voting rather than delegation.

This is the cleanest minimal example of a society-style MAS for beginners.